[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Astrofisica_%28AST%29/Astrofisica_Estelar_%28astro-ph.SR%29/AST-06_Simulacion_Estructura_Estelar_Lane_Emden.ipynb)

# [AST-06] Astrofísica Estelar: Ecuaciones de Estructura Numérica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Astrofísica Estelar: Ecuación del Polítropo de Lane-Emden
# Resolvemos numéricamente la estructura interior de una estrella y 
# encontramos su radio físico (donde la densidad cae al vacío).

# Ecuación Diferencial de 2do orden para la densidad adimensionada Theta:
# (1/xi^2) * d/dxi(xi^2 * dTheta/dxi) = - Theta^n
# La convertimos en un sistema de primer orden para SciPy:
# y[0] = Theta
# y[1] = dTheta/dxi

def lane_emden(xi, y, n):
    theta, dtheta = y
    
    # Evitar la singularidad geométrica en el centro (xi = 0)
    if xi == 0:
        return [0, 0]
    
    # Para prevenir densidades complejas/negativas que detendrían la integración
    theta_safe = max(theta, 0.0) 
    
    d2theta = -theta_safe**n - (2.0/xi)*dtheta
    return [dtheta, d2theta]

# Condiciones iniciales en el núcleo estelar: 
# Densidad máxima (Theta=1) y plana (derivada=0)
y0 = [1.0, 0.0]

# Rango espacial (adimensionado). Integramos desde el núcleo hacia fuera.
# Empezamos ligeramente lejos del 0 matemático para evitar 1/0.
xi_span = (1e-5, 10.0) 
xi_eval = np.linspace(1e-5, 10.0, 500)

plt.figure(figsize=(10, 6))

# Índices politrópicos n (n=0: Incompresible, n=1.5: Enana Blanca, n=3: Sol/Estrella Masiva)
indices_n = [0, 1.5, 3]
colores = ['blue', 'green', 'red']
nombres = ["Planeta rocoso ($n=0$)", "Enana Blanca / Gas ($n=1.5$)", "Sol radiativo ($n=3$)"]

for n, color, nombre in zip(indices_n, colores, nombres):
    # Terminamos la integración cuando la estrella "se acaba" (Densidad Theta = 0)
    def star_surface(t, y, n_val): return y[0]
    star_surface.terminal = True
    star_surface.direction = -1

    sol = solve_ivp(lane_emden, xi_span, y0, args=(n,), t_eval=xi_eval, events=star_surface, method='RK45')
    
    plt.plot(sol.t, sol.y[0], color=color, lw=3, label=f"{nombre}")
    
    # Punto de la Superficie Estelar (Radio R)
    if sol.t_events[0].size > 0:
        xi_surface = sol.t_events[0][0]
        plt.plot(xi_surface, 0, marker='o', color=color, markersize=10)

plt.title("Astrofísica: Estructura Estelar (Ecuación Lane-Emden)", fontsize=14)
plt.xlabel("Radio estelar adimensionado $\xi$ (Distancia desde el núcleo)")
plt.ylabel("Densidad Termodinámica Estelar $\theta(\xi)$")
plt.legend()
plt.grid(True)
plt.xlim(0, 8)
plt.ylim(0, 1.05)
plt.show()

# FÍSICA: En n=3 (una estrella madura de Secuencia Principal), el núcleo está
# brutalmente comprimido y la densidad decae lentamente hasta su superficie.
# Las matemáticas del equilibrio hidrostático no admiten fallo: predicen el radio exacto.

